# nb_test_generator
Unit tests for `nb_lib_generator` (pure Python: no Spark, no lakehouse writes).
Files are generated into a temporary folder on the driver and deleted afterwards.

In [ ]:
%run nb_lib_generator

In [ ]:
import csv
import io
import os
import shutil
import tempfile
from collections import Counter

SMALL = GeneratorConfig(seed=7, history_days=3, initial_customers=50, initial_products=20,
                        orders_per_day=40, dq_error_rate=0.0)
DIRTY = dataclasses.replace(SMALL, orders_per_day=400, dq_error_rate=0.05)
LANDING, STATE = "Files/landing", "Files/_state/generator_state.json"


def tmp_root():
    return tempfile.mkdtemp(prefix="gen_test_")


def read_entity(root, entity):
    """All rows of an entity across all days (csv -> dicts, json -> raw lines)."""
    base = os.path.join(root, LANDING, entity)
    out = []
    for day in sorted(os.listdir(base)):
        for name in os.listdir(os.path.join(base, day)):
            with open(os.path.join(base, day, name), encoding="utf-8") as f:
                text = f.read()
            if name.endswith(".csv"):
                out.extend(csv.DictReader(io.StringIO(text)))
            else:
                out.extend(l for l in text.splitlines() if l)
    return out


def row_key(row):
    """Hashable, comparable key for a CSV row (malformed rows have a None key for extra fields)."""
    return tuple((str(k), str(v)) for k, v in row.items())


def run_tests(namespace):
    results = []
    for name in sorted(k for k, v in namespace.items() if k.startswith("test_") and callable(v)):
        try:
            namespace[name]()
            results.append((name, "PASS", ""))
        except Exception as exc:
            results.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))
    for name, status, msg in results:
        print(f"{'✅' if status == 'PASS' else '❌'} {status}  {name}  {msg}")
    failed = [r for r in results if r[1] == "FAIL"]
    print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
    return failed

## Configuration and driver behaviour

In [ ]:
def test_config_rejects_invalid_values():
    for bad in ({"history_days": 0}, {"orders_per_day": 0}, {"dq_error_rate": 0.5}):
        try:
            GeneratorConfig(**bad)
        except ValueError:
            continue
        raise AssertionError(f"accepted invalid config {bad}")


def test_first_run_generates_history_days():
    root = tmp_root()
    try:
        summary = run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        assert summary["days"] == 3 and summary["files"] == 12, summary
        assert (summary["first_day"], summary["last_day"]) == ("2026-10-31", "2026-11-02")
        assert sorted(os.listdir(os.path.join(root, LANDING, "orders"))) == \
            ["2026-10-31", "2026-11-01", "2026-11-02"]
    finally:
        shutil.rmtree(root)


def test_rerun_for_same_date_does_nothing():
    root = tmp_root()
    try:
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        again = run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        assert again["days"] == 0
    finally:
        shutil.rmtree(root)


def test_catch_up_generates_only_missing_days():
    root = tmp_root()
    try:
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        later = run_generator(root, LANDING, STATE, dt.date(2026, 11, 4), SMALL)
        assert (later["days"], later["first_day"], later["last_day"]) == (2, "2026-11-03", "2026-11-04")
    finally:
        shutil.rmtree(root)


def test_same_inputs_produce_identical_files():
    a, b = tmp_root(), tmp_root()
    try:
        for root in (a, b):
            run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), DIRTY)
        for entity in FILE_FORMATS:
            assert read_entity(a, entity) == read_entity(b, entity), entity
    finally:
        shutil.rmtree(a); shutil.rmtree(b)


def test_seed_mismatch_with_existing_state_fails():
    root = tmp_root()
    try:
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        try:
            run_generator(root, LANDING, STATE, dt.date(2026, 11, 3), dataclasses.replace(SMALL, seed=8))
        except ValueError:
            return
        raise AssertionError("seed change was not detected")
    finally:
        shutil.rmtree(root)

## Data content

In [ ]:
def test_csv_files_have_contract_headers():
    out = to_csv([{"order_id": 1, "customer_id": None}], ORDER_COLS)
    header, row = out.splitlines()
    assert header.split(",") == ORDER_COLS
    assert row.startswith("1,,")  # None -> empty field -> NULL when read by Spark


def test_clean_mode_has_no_injected_problems():
    root = tmp_root()
    try:
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        customers = read_entity(root, "customers")
        assert all("@" in c["email"] and c["email"] == c["email"].strip().lower() for c in customers)
        items = read_entity(root, "order_items")
        product_ids = {json.loads(p)["product_id"] for p in read_entity(root, "products")}
        assert all(int(i["quantity"]) > 0 for i in items)
        assert all(int(i["product_id"]) in product_ids for i in items)
        orders = read_entity(root, "orders")
        for o in orders:
            dt.datetime.strptime(o["order_ts"], TS_FMT)
        for entity in ("customers", "orders", "order_items"):
            rows = [row_key(r) for r in read_entity(root, entity)]
            assert len(rows) == len(set(rows)), f"exact duplicates in {entity}"
    finally:
        shutil.rmtree(root)


def test_dirty_mode_injects_each_problem_type():
    root = tmp_root()
    try:
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), DIRTY)
        orders, items = read_entity(root, "orders"), read_entity(root, "order_items")
        found = {
            "bad_order_ts": any("-02-30" in o["order_ts"] for o in orders),
            "bad_quantity": any(i["quantity"] not in (None, "") and i["quantity"].lstrip("-").isdigit()
                                and int(i["quantity"]) <= 0 for i in items),
            "orphan_product": any(i["product_id"] == str(ORPHAN_PRODUCT_ID) for i in items),
            "duplicate_items": max(Counter(row_key(i) for i in items).values()) > 1,
            "messy_email": any(c["email"] != c["email"].strip() for c in read_entity(root, "customers")),
        }
        missing = [k for k, ok in found.items() if not ok]
        assert not missing, f"not injected: {missing}"
    finally:
        shutil.rmtree(root)


def test_guest_orders_have_empty_customer_and_store_channel():
    root = tmp_root()
    try:
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 2), SMALL)
        guests = [o for o in read_entity(root, "orders") if o["customer_id"] == ""]
        assert guests, "expected some guest orders"
        assert all(o["channel"] == "STORE" for o in guests)
    finally:
        shutil.rmtree(root)


def test_order_status_only_moves_forward():
    allowed = {"PLACED": {"SHIPPED", "CANCELLED"}, "SHIPPED": {"DELIVERED"},
               "DELIVERED": {"RETURNED"}, "COMPLETED": set(), "CANCELLED": set(), "RETURNED": set()}
    root = tmp_root()
    try:
        cfg = dataclasses.replace(SMALL, history_days=10)
        run_generator(root, LANDING, STATE, dt.date(2026, 11, 10), cfg)
        history = {}
        for o in sorted(read_entity(root, "orders"), key=lambda o: o["updated_at"]):
            history.setdefault(o["order_id"], []).append(o["order_status"])
        assert any(len(h) > 2 for h in history.values()), "no multi-step status histories"
        for oid, statuses in history.items():
            for old, new in zip(statuses, statuses[1:]):
                assert new in allowed[old], f"order {oid}: {old} -> {new}"
    finally:
        shutil.rmtree(root)


def test_customer_new_version_changes_one_tracked_attribute():
    tracked = ("city", "loyalty_tier", "segment")
    for cid in range(1, 30):
        v0 = customer_row(1, cid, 0, "2025-01-01", "t0")
        v1 = customer_row(1, cid, 1, "2025-01-01", "t1")
        changed = [a for a in tracked if v0[a] != v1[a]]
        assert len(changed) <= 1, (cid, changed)
        assert (v0["first_name"], v0["email"]) == (v1["first_name"], v1["email"])

## Run

In [ ]:
failed = run_tests(globals())
if failed:
    raise AssertionError(f"{len(failed)} generator test(s) failed: {[f[0] for f in failed]}")